# All-Model Full Metrics Evaluation (Fresh Inference)

This notebook computes metrics from **all available model checkpoints** and builds a single comparison table.

## Computed metrics
- Classification: AUROC, AUPRC, Accuracy, Precision, Recall, Specificity, F1, FPR, FNR
- Reconstruction: MSE, MAE, SSIM (overall + healthy/lesion splits)
- Lesion localization (if masks exist): Dice, IoU

## Outputs
- `evaluations/tables/all_models_full_metrics.csv`
- `evaluations/tables/all_models_full_metrics_ranked.csv`
- `evaluations/tables/all_models_full_metrics_publication.csv`

In [1]:
# Environment and imports (GitHub-first in Colab)
import os
import sys
import importlib.util
import subprocess
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

# Detect Colab
try:
    from google.colab import drive
    IN_COLAB = True
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except Exception:
    IN_COLAB = False

# GitHub-first repo resolution
REPO_URL = 'https://github.com/RifaDeen/symAD-ECNN.git'
COLAB_REPO = Path('/content/symAD-ECNN')

if IN_COLAB:
    if not COLAB_REPO.exists():
        print('Cloning repository from GitHub...')
        subprocess.check_call(['git', 'clone', REPO_URL, str(COLAB_REPO)])
    else:
        print('Repository exists. Pulling latest changes...')
        subprocess.check_call(['git', '-C', str(COLAB_REPO), 'pull'])
    REPO_ROOT = COLAB_REPO
else:
    if Path('C:/Users/rifad/symAD-ECNN').exists():
        REPO_ROOT = Path('C:/Users/rifad/symAD-ECNN')
    else:
        REPO_ROOT = Path.cwd().resolve()

# Project root (Drive preferred when available)
if Path('/content/drive/MyDrive/symAD-ECNN').exists():
    PROJECT_ROOT = Path('/content/drive/MyDrive/symAD-ECNN')
else:
    PROJECT_ROOT = REPO_ROOT

# Ensure module paths
EVALS_DIR = REPO_ROOT / 'notebooks' / 'evals'
ECNN_DIR = EVALS_DIR / 'ecnn_thresholding'
for p in [REPO_ROOT, EVALS_DIR, ECNN_DIR]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Install missing python packages
def ensure_pkg(import_name: str, pip_name: str = None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {pip_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])

ensure_pkg('skimage', 'scikit-image')
ensure_pkg('e2cnn', 'e2cnn')

from skimage.metrics import structural_similarity as ssim

OUT_DIR = PROJECT_ROOT / 'evaluations' / 'tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'IN_COLAB: {IN_COLAB}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'REPO_ROOT: {REPO_ROOT}')
print(f'Device: {device}')

Mounted at /content/drive
Cloning repository from GitHub...
Installing e2cnn...
IN_COLAB: True
PROJECT_ROOT: /content/drive/MyDrive/symAD-ECNN
REPO_ROOT: /content/symAD-ECNN
Device: cuda


In [2]:
# TURBO LOADER (optional): unzip IXI fast zips to local Colab disk
import shutil
from pathlib import Path

data_paths = {}
BASE_DIR = Path('/content/drive/MyDrive/symAD-ECNN/data')
LOCAL_DIR = Path('/content/local_data')

ZIPS_TO_EXTRACT = {
    'train': BASE_DIR / 'train_fast.zip',
    'val':   BASE_DIR / 'val_fast.zip',
    'test':  BASE_DIR / 'test_fast.zip',
}

if Path('/content').exists():
    print('Extracting to Local Disk...')
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)

    for name, zip_path in ZIPS_TO_EXTRACT.items():
        target_dir = LOCAL_DIR / name
        if target_dir.exists():
            shutil.rmtree(target_dir)
        target_dir.mkdir(parents=True, exist_ok=True)

        if zip_path.exists():
            print(f'  Unzipping {name}...')
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(target_dir)
        else:
            print(f'  Skipped (not found): {zip_path}')

    data_paths['ixi_train'] = LOCAL_DIR / 'train'
    data_paths['ixi_val'] = LOCAL_DIR / 'val'
    data_paths['ixi_test'] = LOCAL_DIR / 'test'

    print('\nUpdated data_paths:')
    for key, path in data_paths.items():
        status = 'Found' if path.exists() else 'Not found'
        print(f'  {key}: {status}')
        print(f'    -> {path}')
else:
    print('Not in Colab. Skipping turbo unzip step.')

Extracting to Local Disk...
  Unzipping train...
  Unzipping val...
  Unzipping test...

Updated data_paths:
  ixi_train: Found
    -> /content/local_data/train
  ixi_val: Found
    -> /content/local_data/val
  ixi_test: Found
    -> /content/local_data/test


In [10]:
# BRATS LOADER (explicit): unzip BraTS processed test set and register path
brats_data_path = None

if Path('/content').exists():
    BRATS_BASE = Path('/content/test_eval_data')
    BRATS_ZIP_CANDIDATES = [
        Path('/content/drive/MyDrive/symAD-ECNN/data/brats_test_with_masks_processed.zip'),
        PROJECT_ROOT / 'data' / 'brats_test_with_masks_processed.zip',
    ]

    brats_zip = next((p for p in BRATS_ZIP_CANDIDATES if p.exists()), None)
    if brats_zip is not None:
        print(f'Found BraTS zip: {brats_zip}')
        BRATS_BASE.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(brats_zip, 'r') as zf:
            zf.extractall(BRATS_BASE)

        # support both /processed and direct layout
        for candidate in [BRATS_BASE / 'processed', BRATS_BASE]:
            if (candidate / 'images').exists() and (candidate / 'labels').exists():
                brats_data_path = candidate
                break

        if brats_data_path is not None:
            print(f'BraTS data ready: {brats_data_path}')
        else:
            print('BraTS unzip done, but images/labels structure not found.')
    else:
        print('BraTS zip not found in expected locations.')
else:
    # Local/Windows workspace fallback
    for candidate in [PROJECT_ROOT / 'data' / 'test_eval_data' / 'processed', PROJECT_ROOT / 'data' / 'test_eval_data']:
        if (candidate / 'images').exists() and (candidate / 'labels').exists():
            brats_data_path = candidate
            break
    print(f'BraTS data path (local): {brats_data_path}')

Found BraTS zip: /content/drive/MyDrive/symAD-ECNN/data/brats_test_with_masks_processed.zip
BraTS data ready: /content/test_eval_data


In [ ]:
# Model definitions + checkpoint discovery
def ensure_e2cnn_installed():
    if importlib.util.find_spec('e2cnn') is not None:
        return
    print('Installing e2cnn...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'e2cnn'])

try:
    ensure_e2cnn_installed()
    from ecnn_model_loader import get_model_for_inference
    HAS_ECNN = True
except Exception as e:
    HAS_ECNN = False
    print(f'ECNN loader unavailable: {e}')

from model_defs import CNNAutoencoder, LargeCNNAutoencoder, ResNetAutoencoder
from eval_common import get_state_dict as get_state_dict_common, find_best_checkpoint as find_best_checkpoint_common

CHECKPOINT_ROOTS = [
    PROJECT_ROOT / 'models' / 'saved_models',
    REPO_ROOT / 'models' / 'saved_models',
    Path('/content/drive/MyDrive/symAD-ECNN/models/saved_models')
]
CHECKPOINT_ROOTS = [p for p in CHECKPOINT_ROOTS if p.exists()]
print('Checkpoint roots:')
for p in CHECKPOINT_ROOTS:
    print('-', p)

MODEL_CONFIGS = [
    {'name': 'ECNN-AE (Optimized)', 'key': 'ecnn_opt', 'type': 'ecnn', 'subdirs': ['ecnn_optimized', '.'], 'patterns': ['ecnn_optimized_best.pth', '*ecnn*best*.pth', '*ecnn*.pth']},
    {'name': 'CNN-AE', 'key': 'cnn_base', 'type': 'cnn_base', 'subdirs': ['cnn_ae', '.'], 'patterns': ['cnn_best.pth', '*cnn*best*.pth', '*cnn*.pth']},
    {'name': 'CNN-AE Augmented', 'key': 'cnn_aug', 'type': 'cnn_base', 'subdirs': ['cnn_ae_augmented', '.'], 'patterns': ['*cnn*aug*best*.pth', '*aug*cnn*.pth']},
    {'name': 'CNN-AE Large', 'key': 'cnn_large', 'type': 'cnn_large', 'subdirs': ['cnn_ae_large', '.'], 'patterns': ['*cnn*large*best*.pth', '*large*cnn*.pth']},
    {'name': 'ResNet-AE (frozen)', 'key': 'resnet_ae', 'type': 'resnet_none', 'subdirs': ['resnet_ae', '.'], 'patterns': ['*resnet*best*.pth', '*resnet*.pth']},
    {'name': 'ResNet-AE (partial FT)', 'key': 'resnet_ft_partial', 'type': 'resnet_partial', 'subdirs': ['resnet_finetuned_partial', 'resnet_finetuned', '.'], 'patterns': ['*resnet*best*.pth', '*resnet*.pth']},
    {'name': 'ResNet-AE (full FT)', 'key': 'resnet_ft_full', 'type': 'resnet_full', 'subdirs': ['resnet_finetuned_full', 'resnet_finetuned', '.'], 'patterns': ['*resnet*best*.pth', '*resnet*.pth']},
]

def get_state_dict(ckpt):
    return get_state_dict_common(ckpt)

def find_best_checkpoint(cfg):
    return find_best_checkpoint_common(cfg, CHECKPOINT_ROOTS)

def load_model_from_cfg(cfg):
    ckpt_path = find_best_checkpoint(cfg)
    if ckpt_path is None:
        raise FileNotFoundError('No checkpoint found')

    if cfg['type'] == 'ecnn':
        if not HAS_ECNN:
            raise RuntimeError('ECNN loader unavailable')
        model, _ = get_model_for_inference(str(ckpt_path), str(device))
        return model.to(device).eval(), ckpt_path

    ckpt = torch.load(ckpt_path, map_location=device)
    sd = get_state_dict(ckpt)

    if cfg['type'] == 'cnn_large':
        latent = int(sd['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in sd else 512
        model = LargeCNNAutoencoder(latent_dim=latent)
    elif cfg['type'] == 'cnn_base':
        latent = int(sd['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in sd else 256
        model = CNNAutoencoder(latent_dim=latent)
    elif cfg['type'] == 'resnet_none':
        model = ResNetAutoencoder(finetune_strategy='none')
    elif cfg['type'] == 'resnet_partial':
        model = ResNetAutoencoder(finetune_strategy='partial')
    elif cfg['type'] == 'resnet_full':
        model = ResNetAutoencoder(finetune_strategy='full')
    else:
        raise ValueError(f"Unknown type: {cfg['type']}")

    model.load_state_dict(sd, strict=False)
    return model.to(device).eval(), ckpt_path

Checkpoint roots:
- /content/drive/MyDrive/symAD-ECNN/models/saved_models
- /content/symAD-ECNN/models/saved_models
- /content/drive/MyDrive/symAD-ECNN/models/saved_models


In [ ]:
# Dataset + metric helpers
class EvalDataset(Dataset):
    def __init__(self, image_dir: Path, label_dir: Path, mask_dir=None, mode='grayscale'):
        self.image_files = sorted(image_dir.glob('slice_*.npy'))
        self.label_dir = label_dir
        self.mask_dir = mask_dir if (mask_dir is not None and Path(mask_dir).exists()) else None
        self.mode = mode
        self.resnet_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        sid = img_path.stem.split('_')[-1]
        lab_path = self.label_dir / f'label_{sid}.npy'
        img = np.load(img_path).astype(np.float32)
        label = int(np.load(lab_path)[0])

        gray = torch.from_numpy(img).float().unsqueeze(0)
        target = gray.clone()

        if self.mask_dir is not None:
            mpath = Path(self.mask_dir) / f'slice_{sid}.npy'
            mask = np.load(mpath).astype(np.uint8) if mpath.exists() else np.zeros_like(img, dtype=np.uint8)
        else:
            mask = np.zeros_like(img, dtype=np.uint8)

        if self.mode == 'resnet':
            img_uint8 = (img * 255.0).clip(0, 255).astype(np.uint8)
            rgb = np.stack([img_uint8, img_uint8, img_uint8], axis=-1)
            inp = self.resnet_transform(rgb)
            return inp, target, label, sid, torch.from_numpy(mask).unsqueeze(0).float()

        return gray, target, label, sid, torch.from_numpy(mask).unsqueeze(0).float()

# Primary scoring policy (aligned with thresholding + primary notebooks)
PRIMARY_SCORE_METHOD = 'mean'
PRIMARY_ERROR_MODE = 'squared'
PRIMARY_USE_BRAIN_MASK = True
PRIMARY_MIN_BRAIN_PIXELS = 50
PRIMARY_TARGET_FPR = 0.20
PRIMARY_SCORE_NAME = 'mse_mean_brain_masked'

def normalize01(x):
    x = x.astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx - mn < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn)

def dice_score(pred, gt):
    pred = pred.astype(np.uint8)
    gt = gt.astype(np.uint8)
    inter = (pred * gt).sum()
    return float((2.0 * inter) / (pred.sum() + gt.sum() + 1e-8))

def iou_score(pred, gt):
    pred = pred.astype(np.uint8)
    gt = gt.astype(np.uint8)
    inter = (pred * gt).sum()
    union = ((pred + gt) > 0).sum()
    return float(inter / (union + 1e-8))

def threshold_from_normals(scores, labels, target_fpr=PRIMARY_TARGET_FPR):
    scores = np.asarray(scores, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int32)
    normal_scores = scores[labels == 0]
    if len(normal_scores) == 0:
        return np.nan
    return float(np.percentile(normal_scores, (1 - target_fpr) * 100))

def compute_cls_metrics(labels, scores, threshold):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)
    if scores.size == 0 or labels.size == 0 or np.isnan(threshold):
        return {
            'threshold': float('nan'),
            'auroc': float('nan'),
            'auprc': float('nan'),
            'accuracy': float('nan'),
            'precision': float('nan'),
            'recall': float('nan'),
            'specificity': float('nan'),
            'f1_score': float('nan'),
            'fpr': float('nan'),
            'fnr': float('nan'),
            'tp': 0, 'tn': 0, 'fp': 0, 'fn': 0,
        }

    preds = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold),
        'auroc': float(roc_auc_score(labels, scores)) if np.unique(labels).size > 1 else np.nan,
        'auprc': float(average_precision_score(labels, scores)) if np.unique(labels).size > 1 else np.nan,
        'accuracy': float(accuracy_score(labels, preds)),
        'precision': float(precision_score(labels, preds, zero_division=0)),
        'recall': float(recall_score(labels, preds, zero_division=0)),
        'specificity': float(tn / (tn + fp)) if (tn + fp) > 0 else np.nan,
        'f1_score': float(f1_score(labels, preds, zero_division=0)),
        'fpr': float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan,
        'fnr': float(fn / (fn + tp)) if (fn + tp) > 0 else np.nan,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
    }

def _topk_mean_np(arr_2d: np.ndarray, frac: float) -> np.ndarray:
    n = arr_2d.shape[1]
    k = max(1, int(n * frac))
    part = np.partition(arr_2d, n - k, axis=1)[:, n - k:]
    return part.mean(axis=1)

@torch.no_grad()
def infer_scores_and_lesion_metrics(model, dataloader, pixel_percentile=95):
    score_store = {
        'mse_mean': [], 'mae_mean': [],
        'mse_p95': [], 'mse_p99': [],
        'mae_p95': [], 'mae_p99': [],
        'mse_top1pct_mean': [], 'mae_top1pct_mean': [],
        PRIMARY_SCORE_NAME: [],
    }

    ssim_scores, labels = [], []
    lesion_dice, lesion_iou = [], []
    lesion_count, non_lesion_count = 0, 0

    for x, target, lab, sid, mask in tqdm(dataloader, leave=False):
        x = x.to(device)
        target = target.to(device)
        mask = mask.to(device)

        recon = model(x)
        if recon.shape[-2:] != target.shape[-2:]:
            recon = F.interpolate(recon, size=target.shape[-2:], mode='bilinear', align_corners=False)

        err_abs = torch.abs(recon - target)
        err_sq = (recon - target) ** 2

        err_abs_np = err_abs.detach().cpu().numpy()[:, 0]
        err_sq_np = err_sq.detach().cpu().numpy()[:, 0]
        flat_abs = err_abs_np.reshape(err_abs_np.shape[0], -1)
        flat_sq = err_sq_np.reshape(err_sq_np.shape[0], -1)

        score_store['mse_mean'].extend(flat_sq.mean(axis=1).tolist())
        score_store['mae_mean'].extend(flat_abs.mean(axis=1).tolist())
        score_store['mse_p95'].extend(np.percentile(flat_sq, 95, axis=1).tolist())
        score_store['mse_p99'].extend(np.percentile(flat_sq, 99, axis=1).tolist())
        score_store['mae_p95'].extend(np.percentile(flat_abs, 95, axis=1).tolist())
        score_store['mae_p99'].extend(np.percentile(flat_abs, 99, axis=1).tolist())
        score_store['mse_top1pct_mean'].extend(_topk_mean_np(flat_sq, 0.01).tolist())
        score_store['mae_top1pct_mean'].extend(_topk_mean_np(flat_abs, 0.01).tolist())

        # Primary classification score: mean squared error with optional brain mask
        tgt_np = target.detach().cpu().numpy()[:, 0]
        for b in range(err_sq_np.shape[0]):
            if PRIMARY_USE_BRAIN_MASK:
                brain_mask = tgt_np[b] > 0.01
                if int(brain_mask.sum()) >= int(PRIMARY_MIN_BRAIN_PIXELS):
                    primary_score = float(err_sq_np[b][brain_mask].mean())
                else:
                    # strict policy: skip low-brain slices for primary classification score
                    primary_score = np.nan
            else:
                primary_score = float(err_sq_np[b].mean())
            score_store[PRIMARY_SCORE_NAME].append(primary_score)

        labels.extend(lab.numpy().tolist())

        t_np = target.detach().cpu().numpy()
        r_np = recon.detach().cpu().numpy()
        for b in range(t_np.shape[0]):
            gt_img = t_np[b, 0].astype(np.float32)
            rc_img = r_np[b, 0].astype(np.float32)
            dr = float(max(gt_img.max(), rc_img.max()) - min(gt_img.min(), rc_img.min()))
            dr = dr if dr > 1e-8 else 1.0
            ssim_scores.append(float(ssim(gt_img, rc_img, data_range=dr)))

        err = err_abs.detach().cpu().numpy()
        msk = (mask.detach().cpu().numpy() > 0).astype(np.uint8)
        for b in range(err.shape[0]):
            gt = msk[b, 0]
            if gt.sum() > 0:
                lesion_count += 1
                emap = normalize01(err[b, 0])
                thr = np.percentile(emap, pixel_percentile)
                pred = (emap >= thr).astype(np.uint8)
                lesion_dice.append(dice_score(pred, gt))
                lesion_iou.append(iou_score(pred, gt))
            else:
                non_lesion_count += 1

    labels = np.asarray(labels, dtype=np.int32)
    ssim_scores = np.asarray(ssim_scores, dtype=np.float32)
    score_store = {k: np.asarray(v, dtype=np.float32) for k, v in score_store.items()}

    valid_primary = ~np.isnan(score_store[PRIMARY_SCORE_NAME])
    primary_scores = score_store[PRIMARY_SCORE_NAME][valid_primary]
    primary_labels = labels[valid_primary]

    return {
        'score_store': score_store,
        'mse_scores': score_store['mse_mean'],
        'mae_scores': score_store['mae_mean'],
        'ssim_scores': ssim_scores,
        'primary_scores': primary_scores,
        'primary_labels': primary_labels,
        'labels': labels,
        'primary_valid_count': int(valid_primary.sum()),
        'lesion_dice': lesion_dice,
        'lesion_iou': lesion_iou,
        'lesion_count': int(lesion_count),
        'non_lesion_count': int(non_lesion_count),
    }

In [ ]:
# =========================================================================
# SCENARIO 1: IXI + BraTS (EXCLUSIVELY from Turbo Loader data_paths)
# =========================================================================
import re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# 1. Clean, distinct file loader (NO cross-folder fallback magic)
def _is_image_npy(p: Path):
    if not p.name.lower().endswith('.npy'): return False
    s = str(p).lower().replace('\\', '/')
    if 'label' in p.name.lower() or '/labels/' in s: return False
    if 'mask' in p.name.lower() or '/masks/' in s: return False
    return True

def _sid(p: Path):
    m = re.search(r'(\d+)$', p.stem)
    return m.group(1) if m else None

def load_turbo_records(root: Path, source_name: str, force_label=None):
    if root is None or not Path(root).exists():
        print(f"Directory not found: {root}")
        return []
    root = Path(root)

    image_files = sorted([p for p in root.rglob('*.npy') if _is_image_npy(p)])

    # Quick index for masks and labels
    mask_by_sid, label_by_sid = {}, {}
    for p in root.rglob('*.npy'):
        sid = _sid(p)
        if not sid:
            continue

        s = str(p).lower().replace('\\', '/')
        if 'mask' in p.name.lower() or '/masks/' in s:
            mask_by_sid[sid] = p
        elif force_label is None and ('label' in p.name.lower() or '/labels/' in s):
            try:
                label_by_sid[sid] = int(np.load(p)[0])
            except Exception:
                pass

    records = []
    for ip in image_files:
        sid = _sid(ip)

        lab = force_label
        if lab is None:
            if sid not in label_by_sid:
                # If there's literally no label file, force to lesion so reconstruction metrics can run
                lab = 1
            else:
                lab = label_by_sid[sid]

        if lab not in (0, 1):
            continue

        records.append({
            'image_path': ip,
            'label': lab,
            'mask_path': mask_by_sid.get(sid, None),
            'source': source_name
        })
    return records

# 2. Extract Data Paths strictly from the Turbo Loader variable
val_dir = Path(data_paths['ixi_val']) if 'data_paths' in globals() and 'ixi_val' in data_paths else None
test_dir = Path(data_paths['ixi_test']) if 'data_paths' in globals() and 'ixi_test' in data_paths else None

print("=== TURBO LOADER DATA EXCLUSIVELY ===")
print(f"IXI (val) mapped to  : {val_dir}")
print(f"BraTS (test) mapped to: {test_dir}\n")

turbo_ixi_records = load_turbo_records(val_dir, 'turbo_ixi', force_label=0)
turbo_brats_records = load_turbo_records(test_dir, 'turbo_brats', force_label=None)

print(f"Loaded IXI records (forced label 0)  : {len(turbo_ixi_records)}")
print(f"Loaded BraTS records                 : {len(turbo_brats_records)}")

if len(turbo_brats_records) > 0 and 'label_by_sid' not in locals():
    print("\nNote: Native labels were not found for BraTS in the turbo loader test directory. Assigned fallback 'Tumor' (1) label to allow metrics generation.")

scenario_records = turbo_ixi_records + turbo_brats_records

# Define the MultiSourceEvalDataset locally here for clarity
class MultiSourceEvalDataset(Dataset):
    def __init__(self, records, mode='grayscale'):
        self.records = list(records)
        self.mode = mode
        self.resnet_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        rec = self.records[idx]
        img = np.load(rec['image_path']).astype(np.float32)
        label = int(rec['label'])
        gray = torch.from_numpy(img).float().unsqueeze(0)
        target = gray.clone()
        mask = np.load(rec['mask_path']).astype(np.uint8) if rec['mask_path'] is not None and Path(rec['mask_path']).exists() else np.zeros_like(img, dtype=np.uint8)
        sid = rec['image_path'].stem.split('_')[-1]

        if self.mode == 'resnet':
            img_uint8 = (img * 255.0).clip(0, 255).astype(np.uint8)
            rgb = np.stack([img_uint8, img_uint8, img_uint8], axis=-1)
            inp = self.resnet_transform(rgb)
            return inp, target, label, sid, torch.from_numpy(mask).unsqueeze(0).float()
        return gray, target, label, sid, torch.from_numpy(mask).unsqueeze(0).float()

# 3. Evaluate Phase
if len(scenario_records) > 0:
    print(f"\n===================================================================")
    print(f"EVALUATING TURBO SCENARIO (Total Samples: {len(scenario_records)})")
    print(f"===================================================================")

    results = []

    for cfg in MODEL_CONFIGS:
        print(f"\nEvaluating: {cfg['name']}")
        try:
            model, ckpt_path = load_model_from_cfg(cfg)
            mode = 'resnet' if 'resnet' in cfg['type'] else 'grayscale'

            ds = MultiSourceEvalDataset(scenario_records, mode=mode)
            dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

            out = infer_scores_and_lesion_metrics(model, dl, pixel_percentile=95)

            labels = out['labels']
            mse_scores = out['mse_scores']
            mae_scores = out['mae_scores']
            ssim_scores = out['ssim_scores']

            # Primary policy classification scores
            cls_labels = out['primary_labels']
            cls_scores = out['primary_scores']

            normal_mask = (labels == 0)
            lesion_mask = (labels == 1)

            cls_metrics = compute_cls_metrics(
                cls_labels,
                cls_scores,
                threshold_from_normals(cls_scores, cls_labels, target_fpr=PRIMARY_TARGET_FPR),
            )

            results.append({
                'Model': cfg['name'],
                'Total_Samples': len(labels),
                'Classification_Valid_Samples': int(len(cls_scores)),
                'Selected Score': PRIMARY_SCORE_NAME,
                'Target FPR': PRIMARY_TARGET_FPR,
                'Threshold': cls_metrics['threshold'],
                'MSE (All)': np.mean(mse_scores),
                'MAE (All)': np.mean(mae_scores),
                'SSIM (All)': np.mean(ssim_scores),
                'MSE (Healthy)': np.mean(mse_scores[normal_mask]) if normal_mask.any() else np.nan,
                'MSE (Tumor)': np.mean(mse_scores[lesion_mask]) if lesion_mask.any() else np.nan,
                'SSIM (Healthy)': np.mean(ssim_scores[normal_mask]) if normal_mask.any() else np.nan,
                'SSIM (Tumor)': np.mean(ssim_scores[lesion_mask]) if lesion_mask.any() else np.nan,
                'AUROC': cls_metrics['auroc'],
                'AUPRC': cls_metrics['auprc'],
                'Accuracy': cls_metrics['accuracy'],
                'Precision': cls_metrics['precision'],
                'Recall (Sensitivity)': cls_metrics['recall'],
                'Specificity': cls_metrics['specificity'],
                'F1 Score': cls_metrics['f1_score'],
                'FPR': cls_metrics['fpr'],
                'FNR': cls_metrics['fnr'],
            })
            print(f" -> MSE: {results[-1]['MSE (All)']:.5f} | AUROC: {results[-1]['AUROC']:.4f} | F1: {results[-1]['F1 Score']:.4f}")

        except Exception as e:
            print(f" -> FAILED evaluating {cfg['name']}: {e}")

    if results:
        turbo_df = pd.DataFrame(results).sort_values('AUROC', ascending=False)
        print("\n\n" + "=" * 80)
        print("TURBO SCENARIO (IXI + BraTS) METRICS")
        print("=" * 80)
        display(turbo_df)

        out_path = OUT_DIR / 'turbo_loader_ixi_plus_brats_results.csv'
        turbo_df.to_csv(out_path, index=False)
        print(f"\nSaved metrics to {out_path}")
else:
    print("No records loaded. Please check the turbo unzipper cell inputs.")

=== TURBO LOADER DATA EXCLUSIVELY ===
IXI (val) mapped to  : /content/local_data/val
BraTS (test) mapped to: /content/local_data/test

Loaded IXI records (forced label 0)  : 3652
Loaded BraTS records                 : 7794

Note: Native labels were not found for BraTS in the turbo loader test directory. Assigned fallback 'Tumor' (1) label to allow Reconstruction metrics (MSE/MAE/SSIM) to generate.

EVALUATING TURBO SCENARIO (Total Samples: 11446)

Evaluating: ECNN-AE (Optimized)
Detected ECNNAutoencoderV3 checkpoint. Using latent_dim=1024.


/usr/local/lib/python3.12/dist-packages/e2cnn/nn/modules/r2_conv/basisexpansion_singleblock.py:80: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:37.)
  full_mask[mask] = norms.to(torch.uint8)


Model weights loaded from: /content/drive/MyDrive/symAD-ECNN/models/saved_models/copy_ecnn_optimized/ecnn_optimized_best.pth


  0%|          | 0/358 [00:00<?, ?it/s]

 -> MSE: 0.00464 | MAE: 0.02853 | SSIM: 0.8610

Evaluating: CNN-AE
 -> FAILED evaluating CNN-AE: Error(s) in loading state_dict for CNNAutoencoder:
	size mismatch for encoder.9.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for fc_encode.weight: copying a param with shape torch.Size([1024, 4096]) from checkpoint, the shape in current model is torch.Size([1024, 16384]).
	size mismatch for fc_decode.weight: copying a param with shape torch.Size([4096, 1024]) from checkpoint, the shape in current model is torch.Size([16384, 1024]).
	size mismatch for fc_decode.bias: copying a param with shape torch.Size([4096]) from checkpoint, the shape in current model is torch.Size([16384]).

Evaluating: CNN-AE Augmented


...

 -> MSE: 0.00841 | MAE: 0.04054 | SSIM: 0.7720

Evaluating: ResNet-AE (full FT)


  0%|          | 0/358 [00:00<?, ?it/s]

 -> MSE: 0.00841 | MAE: 0.04054 | SSIM: 0.7720


TURBO SCENARIO (IXI + BraTS) METRICS


,Model,Total_Samples,MSE (All),MAE (All),SSIM (All),MSE (Healthy),MSE (Tumor),SSIM (Healthy),SSIM (Tumor),AUROC (MSE-based),AUPRC (MSE-based)
0,ECNN-AE (Optimized),11446,0.004642,0.028529,0.860998,0.003269,0.005285,0.889961,0.847427,0.774982,0.859627
2,CNN-AE Large,11446,0.007012,0.036451,0.798577,0.005143,0.007888,0.814267,0.791225,0.780275,0.858963
1,CNN-AE Augmented,11446,0.007700,0.038113,0.786061,0.006328,0.008342,0.790067,0.784184,0.707201,0.802416
3,ResNet-AE (frozen),11446,0.008413,0.040545,0.771998,0.006519,0.009301,0.784908,0.765949,0.739841,0.835537
4,ResNet-AE (partial FT),11446,0.008413,0.040545,0.771998,0.006519,0.009301,0.784908,0.765949,0.739841,0.835537
5,ResNet-AE (full FT),11446,0.008413,0.040545,0.771998,0.006519,0.009301,0.784908,0.765949,0.739841,0.835537



Saved metrics to /content/drive/MyDrive/symAD-ECNN/evaluations/tables/turbo_loader_ixi_plus_brats_results.csv


In [ ]:
# =========================================================================
# SCENARIO 2: BraTS-Only Evaluation (Reconstruction + Classification Metrics)
# =========================================================================
import pandas as pd
import numpy as np

print("=== BRATS-ONLY SCENARIO (Classification & Reconstruction) ===")
print(f"BraTS-only root mapped to: {brats_only_root}\n")

# 1. Load BraTS-only strictly from its designated root
brats_only_records = load_turbo_records(brats_only_root, 'brats_only_isolated', force_label=None)
print(f"Loaded BraTS records (labeled 0 or 1): {len(brats_only_records)}")

if len(brats_only_records) > 0:
    print(f"\n===================================================================")
    print(f"EVALUATING BRATS-ONLY SCENARIO (Total Samples: {len(brats_only_records)})")
    print(f"===================================================================")

    brats_results = []

    for cfg in MODEL_CONFIGS:
        print(f"\nEvaluating: {cfg['name']}")
        try:
            model, ckpt_path = load_model_from_cfg(cfg)
            mode = 'resnet' if 'resnet' in cfg['type'] else 'grayscale'

            ds = MultiSourceEvalDataset(brats_only_records, mode=mode)
            dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

            out = infer_scores_and_lesion_metrics(model, dl, pixel_percentile=95)

            labels = out['labels']
            mse_scores = out['mse_scores']
            mae_scores = out['mae_scores']
            ssim_scores = out['ssim_scores']

            normal_mask = (labels == 0)
            lesion_mask = (labels == 1)

            # Primary policy classification score
            cls_labels = out['primary_labels']
            cls_scores = out['primary_scores']

            if np.unique(cls_labels).size > 1:
                thr = threshold_from_normals(cls_scores, cls_labels, target_fpr=PRIMARY_TARGET_FPR)
                cls_metrics = compute_cls_metrics(cls_labels, cls_scores, thr)
                auroc_val = cls_metrics['auroc']
                auprc_val = cls_metrics['auprc']
            else:
                auroc_val, auprc_val = np.nan, np.nan
                cls_metrics = {
                    'accuracy': np.nan, 'precision': np.nan, 'recall': np.nan,
                    'specificity': np.nan, 'f1_score': np.nan, 'threshold': np.nan,
                    'fpr': np.nan, 'fnr': np.nan
                }

            brats_results.append({
                'Model': cfg['name'],
                'Total_Samples': len(labels),
                'Classification_Valid_Samples': int(len(cls_scores)),
                'Selected Score': PRIMARY_SCORE_NAME,
                'Target FPR': PRIMARY_TARGET_FPR,
                'Threshold': cls_metrics['threshold'],
                'MSE (All)': np.mean(mse_scores),
                'MAE (All)': np.mean(mae_scores),
                'SSIM (All)': np.mean(ssim_scores),
                'AUROC': auroc_val,
                'AUPRC': auprc_val,
                'Accuracy': cls_metrics['accuracy'],
                'Precision': cls_metrics['precision'],
                'Recall (Sensitivity)': cls_metrics['recall'],
                'Specificity': cls_metrics['specificity'],
                'F1 Score': cls_metrics['f1_score'],
                'FPR': cls_metrics['fpr'],
                'FNR': cls_metrics['fnr'],
            })

            print(f" -> MSE: {brats_results[-1]['MSE (All)']:.5f} | AUROC: {auroc_val:.4f} | F1: {cls_metrics['f1_score']:.4f}")

        except Exception as e:
            print(f" -> FAILED evaluating {cfg['name']}: {e}")

    if brats_results:
        brats_df = pd.DataFrame(brats_results).sort_values('AUROC', ascending=False)
        print("\n\n" + "=" * 90)
        print("BRATS-ONLY SCENARIO METRICS")
        print("=" * 90)
        display(brats_df)

        out_path = OUT_DIR / 'brats_only_full_metrics_results.csv'
        brats_df.to_csv(out_path, index=False)
        print(f"\nSaved metrics to {out_path}")
else:
    print("No isolated BraTS records cleanly loaded. Please check explicit BraTS zip paths.")

=== BRATS-ONLY SCENARIO (Classification & Reconstruction) ===
BraTS-only root mapped to: /content/test_eval_data

Loaded BraTS records (labeled 0 or 1): 11529

EVALUATING BRATS-ONLY SCENARIO (Total Samples: 11529)

Evaluating: ECNN-AE (Optimized)
Detected ECNNAutoencoderV3 checkpoint. Using latent_dim=1024.
Model weights loaded from: /content/drive/MyDrive/symAD-ECNN/models/saved_models/ecnn_optimized/ecnn_optimized_best.pth


  0%|          | 0/361 [00:00<?, ?it/s]

 -> MSE: 0.00566 | AUROC: 0.4446 | F1: 0.1447

Evaluating: CNN-AE
 -> FAILED evaluating CNN-AE: Error(s) in loading state_dict for CNNAutoencoder:
	size mismatch for encoder.9.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for fc_encode.weight: copying a param with shape torch.Size([1024, 4096]) from checkpoint, the shape in current model is torch.Size([1024, 16384]).
	size mismatch for fc_decode.weight: copying a param with shape torch.Size([4096, 1024]) from checkpoint, the shape in current model is torch.Size([16384, 1024]).
	size mismatch for fc_decode.bias: copying a param with shape torch.Size([4096]) from checkpoint, the shape in current model is torch.Size([16384]).

Evaluating: CNN-AE Augmented


  0%|          | 0/361 [00:00<?, ?it/s]

 -> MSE: 0.00804 | AUROC: 0.5064 | F1: 0.3059

Evaluating: CNN-AE Large


...

 -> MSE: 0.00941 | AUROC: 0.4776 | F1: 0.2174

Evaluating: ResNet-AE (full FT)


  0%|          | 0/361 [00:00<?, ?it/s]

 -> MSE: 0.00941 | AUROC: 0.4776 | F1: 0.2174


BRATS-ONLY SCENARIO METRICS


,Model,Total_Samples,MSE (All),MAE (All),SSIM (All),AUROC,AUPRC,Accuracy,Precision,Recall (Sensitivity),Specificity,F1 Score
1,CNN-AE Augmented,11529,0.008039,0.037859,0.784656,0.506422,0.660073,0.411397,0.647747,0.200187,0.799803,0.305851
2,CNN-AE Large,11529,0.007893,0.038069,0.779008,0.491222,0.640789,0.393269,0.612673,0.172201,0.799803,0.268841
4,ResNet-AE (partial FT),11529,0.009413,0.041331,0.768051,0.477573,0.621148,0.369330,0.554032,0.135244,0.799803,0.217415
3,ResNet-AE (frozen),11529,0.009413,0.041331,0.768051,0.477573,0.621148,0.369330,0.554032,0.135244,0.799803,0.217415
5,ResNet-AE (full FT),11529,0.009413,0.041331,0.768051,0.477573,0.621148,0.369330,0.554032,0.135244,0.799803,0.217415
0,ECNN-AE (Optimized),11529,0.005659,0.030255,0.844524,0.444615,0.592314,0.337757,0.442769,0.086502,0.799803,0.144729



Saved metrics to /content/drive/MyDrive/symAD-ECNN/evaluations/tables/brats_only_full_metrics_results.csv


In [ ]:
# Publication-style tables for BOTH scenarios
if 'results_by_scenario' in globals() and isinstance(results_by_scenario, dict) and len(results_by_scenario) > 0:
    publication_tables = {}

    for scenario_name, df in results_by_scenario.items():
        if df is None or df.empty:
            continue

        pub_cols = [
            'model_name', 'selected_score_name', 'threshold',
            'selected_auroc', 'selected_auprc',
            'accuracy', 'precision', 'recall', 'specificity', 'f1_score',
            'mean_mse_all', 'mean_mae_all', 'mean_ssim_all',
            'mean_mse_normal', 'mean_mse_lesion',
            'mean_mae_normal', 'mean_mae_lesion',
            'mean_ssim_normal', 'mean_ssim_lesion'
        ]
        pub_cols = [c for c in pub_cols if c in df.columns]
        pub_df = df[pub_cols].copy()
        pub_df.insert(0, 'Rank', np.arange(1, len(pub_df) + 1))

        pub_df = pub_df.rename(columns={
            'model_name': 'Model',
            'selected_score_name': 'Selected Score',
            'threshold': 'Threshold',
            'selected_auroc': 'AUROC',
            'selected_auprc': 'AUPRC',
            'accuracy': 'Accuracy',
            'precision': 'Precision',
            'recall': 'Recall',
            'specificity': 'Specificity',
            'f1_score': 'F1 Score',
            'mean_mse_all': 'MSE (All)',
            'mean_mae_all': 'MAE (All)',
            'mean_ssim_all': 'SSIM (All)',
            'mean_mse_normal': 'MSE (Healthy)',
            'mean_mse_lesion': 'MSE (Tumour)',
            'mean_mae_normal': 'MAE (Healthy)',
            'mean_mae_lesion': 'MAE (Tumour)',
            'mean_ssim_normal': 'SSIM (Healthy)',
            'mean_ssim_lesion': 'SSIM (Tumour)'
        })

        publication_tables[scenario_name] = pub_df

        print('\n' + '=' * 100)
        print(f'PUBLICATION TABLE: {scenario_name}')
        print('=' * 100)
        display(pub_df)

        pub_out = OUT_DIR / f'all_models_full_metrics_publication_{scenario_name}.csv'
        pub_df.to_csv(pub_out, index=False)
        print(f'Saved: {pub_out}')
else:
    print('No scenario results available. Run the previous cell first.')

In [ ]:
# Sanity check: verify IXI+BraTS roots are mapped correctly
from pathlib import Path
import numpy as np
import re

def _count_npy_images(root: Path):
    if root is None or not Path(root).exists():
        return 0
    root = Path(root)
    cnt = 0
    for p in root.rglob('*.npy'):
        s = str(p).lower().replace('\\', '/')
        n = p.name.lower()
        if n.startswith('label_') or '/labels/' in s:
            continue
        if 'mask' in n or '/masks/' in s:
            continue
        cnt += 1
    return cnt

def _count_label_files(root: Path):
    if root is None or not Path(root).exists():
        return 0
    root = Path(root)
    labels = list(root.rglob('label_*.npy'))
    if len(labels) == 0:
        labels = [p for p in root.rglob('*.npy') if 'label' in p.name.lower()]
    return len(labels)

def _label_distribution_from_files(root: Path, max_files=2000):
    if root is None or not Path(root).exists():
        return {'0': 0, '1': 0, 'other': 0, 'sampled': 0}
    root = Path(root)
    labels = list(root.rglob('label_*.npy'))
    if len(labels) == 0:
        labels = [p for p in root.rglob('*.npy') if 'label' in p.name.lower()]
    labels = labels[:max_files]
    d0 = d1 = dother = 0
    for lp in labels:
        try:
            v = int(np.load(lp)[0])
            if v == 0:
                d0 += 1
            elif v == 1:
                d1 += 1
            else:
                dother += 1
        except Exception:
            dother += 1
    return {'0': d0, '1': d1, 'other': dother, 'sampled': len(labels)}

def _peek_files(root: Path, limit=5):
    if root is None or not Path(root).exists():
        return []
    root = Path(root)
    files = []
    for p in root.rglob('*.npy'):
        files.append(str(p))
        if len(files) >= limit:
            break
    return files

print('=' * 100)
print('ROOT MAPPING CHECK')
print('=' * 100)
print(f"ixi_mix_root   : {ixi_mix_root}")
print(f"brats_mix_root : {brats_mix_root}")
print(f"brats_only_root: {brats_only_root}")

for name, root in [('IXI side', ixi_mix_root), ('Mix BraTS side', brats_mix_root), ('BraTS-only side', brats_only_root)]:
    print('\n' + '-' * 100)
    print(name)
    exists = (root is not None and Path(root).exists())
    print(f"exists: {exists}")
    if not exists:
        continue
    img_n = _count_npy_images(root)
    lab_n = _count_label_files(root)
    dist = _label_distribution_from_files(root)
    print(f"image-like npy count : {img_n}")
    print(f"label file count      : {lab_n}")
    print(f"label distribution    : {dist}")
    print('example files:')
    for fp in _peek_files(root, limit=5):
        print('  ', fp)

print('\n' + '=' * 100)
print('RECORD COUNTS USED BY PIPELINE')
print('=' * 100)
print(f"ixi_records       : {len(ixi_records) if 'ixi_records' in globals() else 'N/A'}")
print(f"brats_mix_records : {len(brats_mix_records) if 'brats_mix_records' in globals() else 'N/A'}")
print(f"brats_only_records: {len(brats_only_records) if 'brats_only_records' in globals() else 'N/A'}")
if 'dataset_scenarios' in globals():
    print('scenarios:', list(dataset_scenarios.keys()))
else:
    print('scenarios: N/A')

In [9]:
# Dedicated cell to calculate MSE, MAE, and SSIM strictly for IXI+BraTS across all models
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim


def compute_minimal_recon_metrics(model, dataloader):
    mse_list, mae_list, ssim_list = [], [], []

    with torch.no_grad():
        for x, target, _, _, _ in tqdm(dataloader, leave=False, desc="Evaluating"):
            x = x.to(device)
            target = target.to(device)

            recon = model(x)
            # Ensure spatial dimensions match
            if recon.shape[-2:] != target.shape[-2:]:
                recon = F.interpolate(recon, size=target.shape[-2:], mode='bilinear', align_corners=False)

            # Compute pixel-wise MSE/MAE directly
            err_sq = ((recon - target) ** 2).mean(dim=[1,2,3]).cpu().numpy()
            err_abs = torch.abs(recon - target).mean(dim=[1,2,3]).cpu().numpy()

            mse_list.extend(err_sq.tolist())
            mae_list.extend(err_abs.tolist())

            # Compute SSIM slice-by-slice
            t_np = target.cpu().numpy()
            r_np = recon.cpu().numpy()

            for b in range(t_np.shape[0]):
                gt_img = t_np[b, 0].astype(np.float32)
                rc_img = r_np[b, 0].astype(np.float32)
                dr = float(max(gt_img.max(), rc_img.max()) - min(gt_img.min(), rc_img.min()))
                dr = dr if dr > 1e-8 else 1.0
                ssim_list.append(float(ssim(gt_img, rc_img, data_range=dr)))

    return np.mean(mse_list), np.mean(mae_list), np.mean(ssim_list)

if 'scenario_records' in globals() and len(scenario_records) > 0:
    records = scenario_records
    print(f"Running dedicated reconstruction evaluation on IXI+BraTS scenario ({len(records)} samples)...\n")


    recon_results = []

    for cfg in MODEL_CONFIGS:
        print(f"Model: {cfg['name']}")
        try:
            model, _ = load_model_from_cfg(cfg)
            mode = 'resnet' if 'resnet' in cfg['type'] else 'grayscale'

            ds = MultiSourceEvalDataset(records, mode=mode)
            dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)

            mean_mse, mean_mae, mean_ssim = compute_minimal_recon_metrics(model, dl)

            recon_results.append({
                'Model': cfg['name'],
                'MSE': mean_mse,
                'MAE': mean_mae,
                'SSIM': mean_ssim
            })
            print(f" -> MSE: {mean_mse:.6f} | MAE: {mean_mae:.6f} | SSIM: {mean_ssim:.4f}\n")

        except Exception as e:
            print(f" -> ERROR evaluating {cfg['name']}: {e}\n")

    if recon_results:
        recon_df = pd.DataFrame(recon_results).sort_values('MSE', ascending=True).reset_index(drop=True)
        print("=" * 60)
        print("RECONSTRUCTION METRICS (IXI + BraTS)")
        print("=" * 60)
        display(recon_df)

        # Save isolated table
        try:
            out_path = OUT_DIR / 'reconstruction_metrics_ixi_plus_brats_only.csv'
            recon_df.to_csv(out_path, index=False)
            print(f"\nSaved table to {out_path}")
        except Exception:
            pass
else:
    print("Error: 'ixi_plus_brats' not found in dataset_scenarios.")
    print("Please make sure you have successfully run the dataset loading cell above first.")

Running dedicated reconstruction evaluation on IXI+BraTS scenario (11446 samples)...

Model: ECNN-AE (Optimized)
Detected ECNNAutoencoderV3 checkpoint. Using latent_dim=1024.
Model weights loaded from: /content/drive/MyDrive/symAD-ECNN/models/saved_models/copy_ecnn_optimized/ecnn_optimized_best.pth


Evaluating:   0%|          | 0/358 [00:00<?, ?it/s]

 -> MSE: 0.004642 | MAE: 0.028529 | SSIM: 0.8610

Model: CNN-AE
 -> ERROR evaluating CNN-AE: Error(s) in loading state_dict for CNNAutoencoder:
	size mismatch for encoder.9.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for fc_encode.weight: copying a param with shape torch.Size([1024, 4096]) from checkpoint, the shape in current model is torch.Size([1024, 16384]).
	size mismatch for fc_decode.weight: copying a param with shape torch.Size([4096, 1024]) from checkpoint, the shape in current model is torch.Size([16384, 1024]).
	size mismatch for fc_decode.bias: copying a param with shape torch.Size([4096]) from checkpoint, the shape in current model is torch.Size([16384]).

Model: CNN-AE Augmented


Evaluating:   0%|          | 0/358 [00:00<?, ?it/s]

 -> MSE: 0.007700 | MAE: 0.038113 | SSIM: 0.7861

Model: CNN-AE Large


...

 -> MSE: 0.008413 | MAE: 0.040545 | SSIM: 0.7720

Model: ResNet-AE (full FT)


Evaluating:   0%|          | 0/358 [00:00<?, ?it/s]

 -> MSE: 0.008413 | MAE: 0.040545 | SSIM: 0.7720

RECONSTRUCTION METRICS (IXI + BraTS)


,Model,MSE,MAE,SSIM
0,ECNN-AE (Optimized),0.004642,0.028529,0.860998
1,CNN-AE Large,0.007012,0.036451,0.798577
2,CNN-AE Augmented,0.007700,0.038113,0.786061
3,ResNet-AE (full FT),0.008413,0.040545,0.771998
4,ResNet-AE (frozen),0.008413,0.040545,0.771998
5,ResNet-AE (partial FT),0.008413,0.040545,0.771998



Saved table to /content/drive/MyDrive/symAD-ECNN/evaluations/tables/reconstruction_metrics_ixi_plus_brats_only.csv


In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F  # Needed for interpolate

print("===================================================================")
print("EVALUATING COMBINED IXI+BraTS FULL METRICS")
print("===================================================================")

records_to_use = dataset_scenarios.get('ixi_plus_brats', scenario_records)

if not records_to_use:
    print("Error: No combined IXI+BraTS records found. Please ensure `scenario_records` or `dataset_scenarios['ixi_plus_brats']` is populated.")
else:
    print(f"Total samples for combined IXI+BraTS: {len(records_to_use)}")
    combined_results = []

    for cfg in MODEL_CONFIGS:
        print(f"\nEvaluating: {cfg['name']}")
        try:
            model, ckpt_path = load_model_from_cfg(cfg)
            mode = 'resnet' if 'resnet' in cfg['type'] else 'grayscale'

            ds = MultiSourceEvalDataset(records_to_use, mode=mode)
            dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

            out = infer_scores_and_lesion_metrics(model, dl, pixel_percentile=95)

            labels = out['labels']
            normal_mask = (labels == 0)
            lesion_mask = (labels == 1)

            mse_scores_all = out['score_store']['mse_mean']
            mae_scores_all = out['score_store']['mae_mean']
            ssim_scores_all = out['ssim_scores']

            mean_mse_all = np.mean(mse_scores_all)
            mean_mae_all = np.mean(mae_scores_all)
            mean_ssim_all = np.mean(ssim_scores_all)

            mean_mse_healthy = np.mean(mse_scores_all[normal_mask]) if normal_mask.any() else np.nan
            mean_mse_lesion = np.mean(mse_scores_all[lesion_mask]) if lesion_mask.any() else np.nan
            mean_mae_healthy = np.mean(mae_scores_all[normal_mask]) if normal_mask.any() else np.nan
            mean_mae_lesion = np.mean(mae_scores_all[lesion_mask]) if lesion_mask.any() else np.nan
            mean_ssim_healthy = np.mean(ssim_scores_all[normal_mask]) if normal_mask.any() else np.nan
            mean_ssim_lesion = np.mean(ssim_scores_all[lesion_mask]) if lesion_mask.any() else np.nan

            # Primary policy classification score (same as thresholding + primary notebook)
            cls_labels = out['primary_labels']
            cls_scores = out['primary_scores']

            auroc_val, auprc_val = np.nan, np.nan
            cls_metrics = {
                'accuracy': np.nan, 'precision': np.nan, 'recall': np.nan,
                'specificity': np.nan, 'f1_score': np.nan, 'threshold': np.nan,
                'fpr': np.nan, 'fnr': np.nan, 'tp': np.nan, 'tn': np.nan, 'fp': np.nan, 'fn': np.nan,
            }

            if np.unique(cls_labels).size > 1:
                thr = threshold_from_normals(cls_scores, cls_labels, target_fpr=PRIMARY_TARGET_FPR)
                cls_metrics = compute_cls_metrics(cls_labels, cls_scores, thr)
                auroc_val = cls_metrics['auroc']
                auprc_val = cls_metrics['auprc']

            combined_results.append({
                'Model': cfg['name'],
                'Total_Samples': len(labels),
                'Classification_Valid_Samples': int(len(cls_scores)),
                'Selected Score': PRIMARY_SCORE_NAME,
                'Target FPR': PRIMARY_TARGET_FPR,
                'MSE (All)': mean_mse_all,
                'MAE (All)': mean_mae_all,
                'SSIM (All)': mean_ssim_all,
                'MSE (Healthy)': mean_mse_healthy,
                'MSE (Tumor)': mean_mse_lesion,
                'MAE (Healthy)': mean_mae_healthy,
                'MAE (Tumor)': mean_mae_lesion,
                'SSIM (Healthy)': mean_ssim_healthy,
                'SSIM (Tumor)': mean_ssim_lesion,
                'AUROC': auroc_val,
                'AUPRC': auprc_val,
                'Threshold': cls_metrics['threshold'],
                'Accuracy': cls_metrics['accuracy'],
                'Precision': cls_metrics['precision'],
                'Recall (Sensitivity)': cls_metrics['recall'],
                'Specificity': cls_metrics['specificity'],
                'F1 Score': cls_metrics['f1_score'],
                'FPR': cls_metrics['fpr'],
                'FNR': cls_metrics['fnr'],
                'TP': cls_metrics['tp'],
                'TN': cls_metrics['tn'],
                'FP': cls_metrics['fp'],
                'FN': cls_metrics['fn'],
            })
            print(f" -> MSE (All): {mean_mse_all:.5f} | AUROC: {auroc_val:.4f} | F1: {cls_metrics['f1_score']:.4f}")

        except Exception as e:
            print(f" -> FAILED evaluating {cfg['name']}: {e}")

    if combined_results:
        combined_df = pd.DataFrame(combined_results)
        combined_df = combined_df.sort_values(['AUROC', 'F1 Score'], ascending=[False, False]).reset_index(drop=True)

        print("\n\n" + "=" * 80)
        print("COMBINED IXI+BraTS FULL METRICS")
        print("=" * 80)
        display(combined_df)

        out_path = OUT_DIR / 'combined_ixi_brats_full_metrics.csv'
        combined_df.to_csv(out_path, index=False)
        print(f"\nSaved combined metrics to {out_path}")
    else:
        print("No models were successfully evaluated for combined IXI+BraTS metrics.")

EVALUATING COMBINED IXI+BraTS FULL METRICS
Total samples for combined IXI+BraTS: 11446

Evaluating: ECNN-AE (Optimized)
Detected ECNNAutoencoderV3 checkpoint. Using latent_dim=1024.
Model weights loaded from: /content/drive/MyDrive/symAD-ECNN/models/saved_models/copy_ecnn_optimized/ecnn_optimized_best.pth


  0%|          | 0/358 [00:00<?, ?it/s]

 -> MSE (All): 0.00464 | AUROC (Top1pct MSE): 0.8017 | F1: 0.7008

Evaluating: CNN-AE
 -> FAILED evaluating CNN-AE: Error(s) in loading state_dict for CNNAutoencoder:
	size mismatch for encoder.9.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for fc_encode.weight: copying a param with shape torch.Size([1024, 4096]) from checkpoint, the shape in current model is torch.Size([1024, 16384]).
	size mismatch for fc_decode.weight: copying a param with shape torch.Size([4096, 1024]) from checkpoint, the shape in current model is torch.Size([16384, 1024]).
	size mismatch for fc_decode.bias: copying a param with shape torch.Size([4096]) from checkpoint, the shape in current model is torch.Size([16384]).

Evaluating: CNN-AE Augmented


  0%|          | 0/358 [00:00<?, ?it/s]

 -> MSE (All): 0.00770 | AUROC (Top1pct MSE): 0.7259 | F1: 0.5308

Evaluating: CNN-AE Large


...

 -> MSE (All): 0.00841 | AUROC (Top1pct MSE): 0.7525 | F1: 0.6013

Evaluating: ResNet-AE (full FT)


  0%|          | 0/358 [00:00<?, ?it/s]

 -> MSE (All): 0.00841 | AUROC (Top1pct MSE): 0.7525 | F1: 0.6013


COMBINED IXI+BraTS FULL METRICS


,Model,Total_Samples,MSE (All),MAE (All),SSIM (All),MSE (Healthy),MSE (Tumor),MAE (Healthy),MAE (Tumor),SSIM (Healthy),...,Precision,Recall (Sensitivity),Specificity,F1 Score,FPR,FNR,TP,TN,FP,FN
0,ECNN-AE (Optimized),11446,0.004642,0.028529,0.860998,0.003269,0.005285,0.025022,0.030172,0.889961,...,0.862826,0.589941,0.799836,0.700754,0.200164,0.410059,4598,2921,731,3196
1,CNN-AE Large,11446,0.007012,0.036451,0.798577,0.005143,0.007888,0.032419,0.038341,0.814267,...,0.854411,0.550423,0.799836,0.669528,0.200164,0.449577,4290,2921,731,3504
2,ResNet-AE (partial FT),11446,0.008413,0.040545,0.771998,0.006519,0.009301,0.037163,0.042130,0.784908,...,0.833712,0.470234,0.799836,0.601313,0.200164,0.529766,3665,2921,731,4129
3,ResNet-AE (full FT),11446,0.008413,0.040545,0.771998,0.006519,0.009301,0.037163,0.042130,0.784908,...,0.833712,0.470234,0.799836,0.601313,0.200164,0.529766,3665,2921,731,4129
4,ResNet-AE (frozen),11446,0.008413,0.040545,0.771998,0.006519,0.009301,0.037163,0.042130,0.784908,...,0.833712,0.470234,0.799836,0.601313,0.200164,0.529766,3665,2921,731,4129
5,CNN-AE Augmented,11446,0.007700,0.038113,0.786061,0.006328,0.008342,0.035665,0.039260,0.790067,...,0.808187,0.395176,0.799836,0.530806,0.200164,0.604824,3080,2921,731,4714



Saved combined metrics to /content/drive/MyDrive/symAD-ECNN/evaluations/tables/combined_ixi_brats_full_metrics.csv
